In [1]:
##############################################################
### IMPORT ALL PACKAGES: MAKE SURE TO USE THE SE3nv KERNEL ###
##############################################################

# Standard library imports
import glob
import json
import math
import os
import random
import re
import shutil
import statistics
import string
import subprocess
import sys
import textwrap
import warnings
from collections import Counter, defaultdict, OrderedDict
import concurrent.futures 
from concurrent.futures import ThreadPoolExecutor
from itertools import permutations

# Third-party imports
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import pyrosetta
import pyrosetta.distributed.tasks.rosetta_scripts as rosetta_scripts
from scipy.stats import gaussian_kde

# Local application/library specific imports
os.chdir('/home/woodbuse/eosin/scripts')
from SimplePdbLib import *
import analyze_chemnet_sethW_script 
import jupyter_utils_v2
import rainier
import util
from make_mpnn_OMIT_jsonl_for_iterativeMPNN import (
    add_filtered_donor_residues_column,
    add_new_omit_AAs_from_filtered_df,
    make_new_omit_AA_jsonl_from_filtered_df,
    optimal_process_pdb_names_efficient_and_logged,
    process_pdb_names,
    modernized_add_new_omit_AAs_from_filtered_df_function
)
from make_sub_script import make_af2_submit_file
from pyfastrelax_stat_processing import parse_energy_tables

# Good colors definition (assuming these are used within the script)
good_teal = (40/255, 176/255, 193/255)
good_yellow = (250/255, 199/255, 44/255)
good_green = (170/255, 195/255, 47/255)
good_pink = (236/255, 114/255, 164/255)
good_peach = (249/255, 145/255, 120/255)
good_gray = (220/255, 220/255, 220/255)
good_blue = (68/255, 153/255, 231/255)
good_red = (228/255, 74/255, 62/255)

# Apply matplotlib style
plt.style.use('/home/woodbuse/eosin/scripts/plot_prefs.mplstyle')

display_all = True  # Set this to False if you want to revert to default display settings

# Set pandas display options based on the display_all variable
if display_all:
    pd.set_option('display.max_columns', None)  # Display all columns
    pd.set_option('display.max_rows', None)  # Display all rows
    pd.set_option('display.max_colwidth', None)  # Display full content of each cell
else:
    pd.reset_option('display.max_columns')
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_colwidth')

In [2]:
# Define base directory for output, adjust as needed
analysis_dir = "/home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/"
params_file = "/home/woodbuse/carbonic_anhydrase/theozymes/HHH_to_Zn/znO_thr_COO_deprot/CZN.params"

### CONSTANTS ###
script_path = "/home/woodbuse/special_scripts/invrot/invrot_analysis_functions_Seth.py"
container_path = "/software/containers/crispy.sif"  # Path to the Singularity container

# Atom pairs for distance measurement
atom_pairs = {
    "amideN_carbondioxO3_dis_IMPORTANT": {
        "atom1": {"chain": "E", "residue": 5, "atom_name": "N"},
        "atom2": {"chain": "F", "residue": 6, "atom_name": "O3"}
    },
    "amideN_carbondioxO2_dis": {
        "atom1": {"chain": "F", "residue": 6, "atom_name": "O2"},
        "atom2": {"chain": "E", "residue": 5, "atom_name": "N"}
    }
}

# Convert atom pairs dictionary to JSON and save to .json file
atom_pairs_json_path = os.path.join(analysis_dir, "input_arguments.json")
with open(atom_pairs_json_path, 'w') as json_file:
    json.dump(atom_pairs, json_file)

# Format the command string to execute the script using Singularity
command = f"singularity exec {container_path} python {script_path} {analysis_dir} {atom_pairs_json_path} {params_file}"

# Print the navigation and command instructions
print("NAVIGATE HERE:")
print(f'cd {analysis_dir}')
print('')
print('COMMAND:')
print(command)
print('')
print('OUTPUT FILE:')
print(f'{analysis_dir}invrot_analysis.csv')

NAVIGATE HERE:
cd /home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/

COMMAND:
singularity exec /software/containers/crispy.sif python /home/woodbuse/special_scripts/invrot/invrot_analysis_functions_Seth.py /home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/ /home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/input_arguments.json /home/woodbuse/carbonic_anhydrase/theozymes/HHH_to_Zn/znO_thr_COO_deprot/CZN.params

OUTPUT FILE:
/home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/invrot_analysis.csv


In [18]:
### Configuration Section ###
csv_file_path = '/home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/invrot_analysis.csv'
distance1 = 3.6
distance2 = 7
sorting_variable = "amideN_carbondioxO3_dis_IMPORTANT"

### Data Loading Section ###
df = pd.read_csv(csv_file_path)

### Data Filtering Section ###
mask = (df['amideN_carbondioxO3_dis_IMPORTANT'] <= distance1) & (df['amideN_carbondioxO2_dis'] <= distance2)
filtered_df = df[mask]

### Calculate the filtering rate and display it ###
mask_filter_rate = float(len(filtered_df)) / float(len(df)) * 100.0
print(f'{len(filtered_df)} of {len(df)} ({round(mask_filter_rate, 3)}%) satisfy distance requirements.')

### Data Processing Section ###
# Sort the DataFrame based on the specified sorting variable
filtered_df.sort_values(by=sorting_variable, ascending=True, inplace=True)
print(filtered_df.head())

### Output Section ###
print('')

# Select 10 random pdb_path from the filtered DataFrame (using .sample if uncommented)
random_pdb_paths = filtered_df['pdb_path'].tolist()  # .sample(n=20, random_state=4)

# Prepare and print the pymol command string with the selected pdb paths
pymol_command = "pymol " + " ".join(random_pdb_paths)
print(pymol_command)


234 of 954 (24.528%) satisfy distance requirements.
     amideN_carbondioxO3_dis_IMPORTANT  amideN_carbondioxO2_dis  \
1                             3.203486                 3.391737   
636                           3.203486                 3.391737   
640                           3.203486                 3.391737   
647                           3.203486                 3.391737   
650                           3.203486                 3.391737   

                                                                                                pdb_path  
1    /home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/CZN_H_H_H_E_T_1_22541.pdb  
636  /home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/CZN_H_H_H_E_T_1_15181.pdb  
640  /home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/CZN_H_H_H_E_T_1_11821.pdb  
647  /home/woodbuse/carbonic_anhydrase/invrot/hhh_to_zn/znO_thr_COO_deprot_CZN/CZN_H_H_H_E_T_1_13301.pdb  
650  /home

/scratch/woodbuse/37234553/ipykernel_375413/382004980.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.sort_values(by=sorting_variable, ascending=True, inplace=True)
